In [ ]:
import torch, pandas as pd, numpy as np, joblib
from src.data_prep import load_properties, merge_scenario_data, ScenarioDataset
from src.model import DeepSetsPredictor
from src.paths import PROPS_PATH, TEST_PATH, MODEL_PATH

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
meta = torch.load('scalers.pkl', map_location='cpu', weights_only=False)
model = DeepSetsPredictor(n_props=len(meta['prop_cols'])).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False))
model.eval()

In [ ]:
props_df = load_properties(PROPS_PATH)
test_df = pd.read_csv(TEST_PATH)
merged_test, p_cols = merge_scenario_data(test_df, props_df)
test_ds = ScenarioDataset(merged_test, p_cols, fit_scaler=False)
# Подставляем скейлеры
test_ds.prop_scaler = meta['prop_scaler']
test_ds.cond_scaler = meta['cond_scaler']
test_ds.target_scaler = meta['target_scaler']

In [ ]:
loader = torch.utils.data.DataLoader(test_ds, batch_size=64, shuffle=False)
vis_preds, ox_preds = [], []
with torch.no_grad():
    for batch in loader:
        p = batch['props'].to(DEVICE)
        m = batch['mask'].to(DEVICE)
        c = batch['conc'].to(DEVICE)
        cond = batch['conditions'].to(DEVICE)
        v, o = model(p, m, c, cond)
        vis_preds.append(v.cpu().numpy())
        ox_preds.append(o.cpu().numpy())

# Инверсия скейлинга
vis_preds = meta['target_scaler'].inverse_transform(
    np.column_stack([np.concatenate(vis_preds), np.zeros_like(np.concatenate(vis_preds))]
))[:,0]
ox_preds = meta['target_scaler'].inverse_transform(
    np.column_stack([np.zeros_like(np.concatenate(ox_preds)), np.concatenate(ox_preds)]
))[:,1]

In [ ]:
res = pd.DataFrame({
    'scenario_id': test_ds.scenario_ids,
    'Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %': vis_preds,
    'Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm': ox_preds
})
res.to_csv('predictions.csv', index=False)
print("predictions.csv готов. Проверка формата...")
assert set(res['scenario_id']) == set(pd.read_csv('daimler_mixtures_test.csv')['scenario_id'].unique())
print("Все 40 сценария присутствуют. Готово к загрузке.")